In [ ]:
!sudo apt-get install parallel

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  sysstat
Suggested packages:
  ash csh fish ksh tcsh zsh isag
The following NEW packages will be installed:
  parallel sysstat
0 upgraded, 2 newly installed, 0 to remove and 35 not upgraded.
Need to get 2,434 kB of archives.
After this operation, 4,521 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 sysstat amd64 12.5.2-2ubuntu0.2 [487 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 parallel all 20210822+ds-2 [1,947 kB]
Fetched 2,434 kB in 2s (1,555 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!find /content/drive/MyDrive/Fum/term9/Project_phase1/* -type f | parallel -j64 rsync -R {} /content/

In [ ]:
!cp -r /content/drive/MyDrive/Fum/term9/Project_phase1/* /content/

In [ ]:
!cp -r /content/drive/MyDrive/Fum/term9 /content/

In [ ]:
!cp -r /content/term9/* /content/

In [ ]:
!cp -r /content/drive/MyDrive/Fum/term9/ocr_finetuned.pth /content/

In [ ]:
!cp -r /content/drive/MyDrive/Fum/term9/dataset_hr /content/

In [ ]:
!cp -r /content/Project_phase1/* /content/

In [ ]:
%%writefile /content/model.py

import torch
import torch.nn as nn
import json

# تنظیمات
num_classes = 35
hidden_size = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ocr_model_path = "/content/ocr_finetuned.pth"
vocab_path = "/content/vocab.json"

# لود وکبیولری
with open(vocab_path, 'r', encoding='utf-8') as f:
    vocab = json.load(f)
idx_to_char = {v: k for k, v in vocab.items()}

# تابع برای بررسی وزن‌های فایل .pth
def inspect_weights(pth_file):
    print(f"📋 بررسی وزن‌های فایل {pth_file}...")
    state_dict = torch.load(pth_file, map_location='cpu')
    print("کلیدها و ابعاد وزن‌ها:")
    for key, value in state_dict.items():
        print(f"  {key}: {list(value.shape)}")
    return state_dict

import torch
import torch.nn as nn

# تعریف کلاس کمکی برای چاپ شکل تنسورها
class PrintShape(nn.Module):
    def __init__(self, layer_name):
        super(PrintShape, self).__init__()
        self.layer_name = layer_name

    def forward(self, x):
        #print(f"شکل خروجی بعد از لایه {self.layer_name}: {x.shape}")
        return x

# تعریف مدل CRNN
class CRNN(nn.Module):
    def __init__(self, num_classes, hidden_size=256):
        super(CRNN, self).__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            PrintShape("Conv1"),
            nn.ReLU(True),
            PrintShape("ReLU1"),
            nn.Dropout2d(0.3),
            PrintShape("Dropout1"),
            nn.MaxPool2d(kernel_size=2, stride=2),
            PrintShape("MaxPool1"),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            PrintShape("Conv2"),
            nn.ReLU(True),
            PrintShape("ReLU2"),
            nn.Dropout2d(0.3),
            PrintShape("Dropout2"),
            nn.MaxPool2d(kernel_size=2, stride=2),
            PrintShape("MaxPool2"),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            PrintShape("Conv3"),
            nn.ReLU(True),
            PrintShape("ReLU3"),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            PrintShape("Conv4"),
            nn.ReLU(True),
            PrintShape("ReLU4"),
            nn.Dropout2d(0.3),
            PrintShape("Dropout3"),
            nn.MaxPool2d(kernel_size=2, stride=2),
            PrintShape("MaxPool3"),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            PrintShape("Conv5"),
            nn.ReLU(True),
            PrintShape("ReLU5"),
            nn.BatchNorm2d(512),
            PrintShape("BatchNorm"),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            PrintShape("Conv6"),
            nn.ReLU(True),
            PrintShape("ReLU6"),
            nn.Dropout2d(0.3),
            PrintShape("Dropout4"),
            nn.MaxPool2d(kernel_size=(4, 1)),
            PrintShape("MaxPool4"),
            nn.Conv2d(512, 512, kernel_size=2, stride=1),
            PrintShape("Conv7"),
            nn.ReLU(True),
            PrintShape("ReLU7")
        )

        self.dropout_rnn = nn.Dropout(0.2)

        self.rnn = nn.Sequential(
            nn.LSTM(input_size=512, hidden_size=hidden_size, num_layers=2,
                    bidirectional=True, batch_first=True)
        )

        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        conv = self.cnn(x)
        batch_size, channels, height, width = conv.size()
        conv = conv.squeeze(2)
        conv = conv.permute(0, 2, 1)
        conv = self.dropout_rnn(conv)
        rnn_out, _ = self.rnn(conv)
        out = self.fc(rnn_out)
        return out

# تابع برای چک کردن سازگاری مدل با وزن‌ها
def check_model_compatibility(model, state_dict):
    print("\n🔍 بررسی سازگاری مدل با وزن‌ها...")
    model_state_dict = model.state_dict()
    missing_keys = [key for key in model_state_dict if key not in state_dict]
    unexpected_keys = [key for key in state_dict if key not in model_state_dict]
    mismatched_shapes = []

    for key in model_state_dict:
        if key in state_dict:
            if model_state_dict[key].shape != state_dict[key].shape:
                mismatched_shapes.append((key, model_state_dict[key].shape, state_dict[key].shape))

    if missing_keys:
        print(f"❌ کلیدهای گم‌شده در state_dict: {missing_keys}")
    else:
        print("✅ هیچ کلید گم‌شده‌ای وجود ندارد")

    if unexpected_keys:
        print(f"❌ کلیدهای غیرمنتظره در state_dict: {unexpected_keys}")
    else:
        print("✅ هیچ کلید غیرمنتظره‌ای وجود ندارد")

    if mismatched_shapes:
        print("❌ ناسازگاری در ابعاد وزن‌ها:")
        for key, model_shape, state_shape in mismatched_shapes:
            print(f"  {key}: مدل={model_shape}, فایل وزن‌ها={state_shape}")
    else:
        print("✅ همه ابعاد وزن‌ها سازگار هستند")

# لود و بررسی وزن‌ها
state_dict = inspect_weights(ocr_model_path)

# تعریف و بررسی مدل
ocr_model = CRNN(num_classes=num_classes, hidden_size=hidden_size).to(device)
check_model_compatibility(ocr_model, state_dict)

# تلاش برای لود مدل
try:

    ocr_model.load_state_dict(state_dict)
    ocr_model.eval()
    print(f"✅ مدل OCR با موفقیت از {ocr_model_path} لود شد!")
except RuntimeError as e:
    print(f"❌ خطا در لود کردن مدل: {e}")

# تعریف لاس CTC (برای استفاده بعدی)
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)

# تابع دیکدینگ
def ctc_decode(output, idx_to_char):
    output = output.softmax(2)
    output = output.argmax(2).squeeze(0).detach().cpu().numpy()
    pred_str = []
    last = -1
    for ch in output:
        if ch != last and ch != 0:
            pred_str.append(idx_to_char[ch])
        last = ch
    return ''.join(pred_str)

Writing /content/model.py


In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from PIL import Image
import os
import json
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from model import CRNN

num_classes = 35
hidden_size = 256
batch_size = 8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ocr_model_path = "/content/ocr_finetuned.pth"
vocab_path = "/content/vocab.json"
dataset_path = "/content/dataset_phase1"
labels_csv = "/content/labels.csv"

with open(vocab_path, 'r', encoding='utf-8') as f:
    vocab = json.load(f)
idx_to_char = {v: k for k, v in vocab.items()}

class Phase1Dataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.folder_to_images = {}
        for folder_idx in range(1, 301):
            folder_path = os.path.join(root_dir, str(folder_idx))
            if os.path.exists(folder_path):
                images = [f for f in os.listdir(folder_path) if f.endswith('.png')]
                if len(images) == 5:
                    self.folder_to_images[str(folder_idx)] = images

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        folder_name = row['image'].split('.')[0]
        label = row['plate']
        folder_path = os.path.join(self.root_dir, folder_name)
        images = []
        for img_name in self.folder_to_images[folder_name]:
            img_path = os.path.join(folder_path, img_name)
            image = Image.open(img_path).convert('L')
            if self.transform:
                image = self.transform(image)
            images.append(image)
        images = torch.stack(images)
        return images, label

val_transform = transforms.Compose([
    transforms.Resize((64, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

dataset = Phase1Dataset(labels_csv, dataset_path, transform=val_transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

ocr_model = CRNN(num_classes=num_classes, hidden_size=hidden_size).to(device)
ocr_model.load_state_dict(torch.load(ocr_model_path, map_location=device))
ocr_model.eval()

def ctc_decode(output, idx_to_char):
    output = output.softmax(2)
    output = output.argmax(2).detach().cpu().numpy()
    pred_str = []
    last = -1
    for ch in output[0]:
        if ch != last and ch != 0:
            pred_str.append(idx_to_char[ch])
        last = ch
    return ''.join(pred_str)

def evaluate_ocr_model(model, dataloader, idx_to_char):
    model.eval()
    total_correct_plates = 0
    total_correct_chars = 0
    total_chars = 0
    total_samples = 0

    with torch.no_grad():
        for batch_images, batch_labels in dataloader:
            batch_images = batch_images.to(device)
            batch_size = batch_images.size(0)
            for i in range(batch_size):
                images = batch_images[i]
                true_label = batch_labels[i]
                for j in range(5):
                    image = images[j:j+1]
                    output = model(image)
                    pred = ctc_decode(output, idx_to_char)
                    if pred == true_label:
                        total_correct_plates += 1
                    min_len = min(len(pred), len(true_label))
                    correct_chars = sum(1 for p, g in zip(pred[:min_len], true_label[:min_len]) if p == g)
                    total_correct_chars += correct_chars
                    total_chars += len(true_label)
                    total_samples += 1

    plate_accuracy = total_correct_plates / total_samples * 100
    char_accuracy = total_correct_chars / total_chars * 100
    return plate_accuracy, char_accuracy

plate_acc, char_acc = evaluate_ocr_model(ocr_model, dataloader, idx_to_char)

print(f"دقت پلاک کامل: {plate_acc:.2f}%")
print(f"دقت کاراکتری: {char_acc:.2f}%")

📋 بررسی وزن‌های فایل /content/ocr_finetuned.pth...
کلیدها و ابعاد وزن‌ها:
  cnn.0.weight: [64, 1, 3, 3]
  cnn.0.bias: [64]
  cnn.8.weight: [128, 64, 3, 3]
  cnn.8.bias: [128]
  cnn.16.weight: [256, 128, 3, 3]
  cnn.16.bias: [256]
  cnn.20.weight: [256, 256, 3, 3]
  cnn.20.bias: [256]
  cnn.28.weight: [512, 256, 3, 3]
  cnn.28.bias: [512]
  cnn.32.weight: [512]
  cnn.32.bias: [512]
  cnn.32.running_mean: [512]
  cnn.32.running_var: [512]
  cnn.32.num_batches_tracked: []
  cnn.34.weight: [512, 512, 3, 3]
  cnn.34.bias: [512]
  cnn.42.weight: [512, 512, 2, 2]
  cnn.42.bias: [512]
  rnn.0.weight_ih_l0: [1024, 512]
  rnn.0.weight_hh_l0: [1024, 256]
  rnn.0.bias_ih_l0: [1024]
  rnn.0.bias_hh_l0: [1024]
  rnn.0.weight_ih_l0_reverse: [1024, 512]
  rnn.0.weight_hh_l0_reverse: [1024, 256]
  rnn.0.bias_ih_l0_reverse: [1024]
  rnn.0.bias_hh_l0_reverse: [1024]
  rnn.0.weight_ih_l1: [1024, 512]
  rnn.0.weight_hh_l1: [1024, 256]
  rnn.0.bias_ih_l1: [1024]
  rnn.0.bias_hh_l1: [1024]
  rnn.0.weight_ih_

In [ ]:
pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.5/963.5 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 820.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 43.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [ ]:
pip install torchvision scipy

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from PIL import Image
import os
import json
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from model import CRNN

num_classes = 35
hidden_size = 256
batch_size = 8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ocr_model_path = "/content/ocr_finetuned.pth"
vocab_path = "/content/vocab.json"
dataset_path = "/content/dataset_phase1"
labels_csv = "/content/labels.csv"

with open(vocab_path, 'r', encoding='utf-8') as f:
    vocab = json.load(f)
idx_to_char = {v: k for k, v in vocab.items()}

class Phase1Dataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.folder_to_images = {}
        for folder_idx in range(1, 301):
            folder_path = os.path.join(root_dir, str(folder_idx))
            if os.path.exists(folder_path):
                images = [f for f in os.listdir(folder_path) if f.endswith('.png')]
                if len(images) == 5:
                    self.folder_to_images[str(folder_idx)] = images

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        folder_name = row['image'].split('.')[0]
        label = row['plate']
        folder_path = os.path.join(self.root_dir, folder_name)
        images = []
        for img_name in self.folder_to_images[folder_name]:
            img_path = os.path.join(folder_path, img_name)
            image = Image.open(img_path).convert('L')
            if self.transform:
                image = self.transform(image)
            images.append(image)
        images = torch.stack(images)
        return images, label

val_transform = transforms.Compose([
    transforms.Resize((64, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

dataset = Phase1Dataset(labels_csv, dataset_path, transform=val_transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

ocr_model = CRNN(num_classes=num_classes, hidden_size=hidden_size).to(device)
ocr_model.load_state_dict(torch.load(ocr_model_path, map_location=device))
ocr_model.eval()

def ctc_decode(output, idx_to_char):
    output = output.softmax(2)
    output = output.argmax(2).detach().cpu().numpy()
    pred_str = []
    last = -1
    for ch in output[0]:
        if ch != last and ch != 0:
            pred_str.append(idx_to_char[ch])
        last = ch
    return ''.join(pred_str)

def evaluate_ocr_model(model, dataloader, idx_to_char):
    model.eval()
    total_correct_plates = 0
    total_correct_chars = 0
    total_chars = 0
    total_samples = 0

    with torch.no_grad():
        for batch_images, batch_labels in dataloader:
            batch_images = batch_images.to(device)
            batch_size = batch_images.size(0)
            for i in range(batch_size):
                images = batch_images[i]
                true_label = batch_labels[i]
                for j in range(5):
                    image = images[j:j+1]
                    output = model(image)
                    pred = ctc_decode(output, idx_to_char)
                    if pred == true_label:
                        total_correct_plates += 1
                    min_len = min(len(pred), len(true_label))
                    correct_chars = sum(1 for p, g in zip(pred[:min_len], true_label[:min_len]) if p == g)
                    total_correct_chars += correct_chars
                    total_chars += len(true_label)
                    total_samples += 1

    plate_accuracy = total_correct_plates / total_samples * 100
    char_accuracy = total_correct_chars / total_chars * 100
    return plate_accuracy, char_accuracy

plate_acc, char_acc = evaluate_ocr_model(ocr_model, dataloader, idx_to_char)

print(f"دقت پلاک کامل: {plate_acc:.2f}%")
print(f"دقت کاراکتری: {char_acc:.2f}%")

دقت پلاک کامل: 70.87%
دقت کاراکتری: 94.14%


In [ ]:
pip install clean-fid

In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils import spectral_norm

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            spectral_norm(nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1)),  # Increased to 32
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.2),
            spectral_norm(nn.Conv2d(32, 96, kernel_size=4, stride=2, padding=1)),  # Increased to 96
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.2),
            spectral_norm(nn.Conv2d(96, 192, kernel_size=4, stride=2, padding=1)),  # Increased to 192
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.2),
            spectral_norm(nn.Conv2d(192, 384, kernel_size=4, stride=2, padding=1)),  # Increased to 384
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.2),
            spectral_norm(nn.Conv2d(384, 384, kernel_size=4, stride=2, padding=1)),  # Increased to 384
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(384, 1, kernel_size=2, stride=1, padding=0)),
            nn.Flatten()
        )
        self._initialize_weights()
        self._count_parameters()

    def forward(self, x):
        return self.model(x)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu', a=0.2)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def _count_parameters(self):
        total_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Total number of trainable parameters: {total_params:,}")

In [ ]:

import torch
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, in_channels=5):
        super(Generator, self).__init__()

        def conv_block(in_channels, out_channels, kernel_size=3, stride=1, padding=1, use_bn=True):
            layers = [
                nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=not use_bn),
                nn.BatchNorm2d(out_channels) if use_bn else nn.Identity(),
                nn.LeakyReLU(0.2, inplace=True),
                nn.Dropout2d(p=0.3),  # اضافه کردن Dropout2d برای فیوژن بهتر
            ]
            return nn.Sequential(*layers)

        def upsample_block(in_channels, out_channels, kernel_size=3, stride=1, padding=1, use_bn=True):
            layers = [
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
                nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=not use_bn),
                nn.BatchNorm2d(out_channels) if use_bn else nn.Identity(),
                nn.LeakyReLU(0.2, inplace=True),
                nn.Dropout2d(p=0.3),  # اضافه کردن Dropout2d
            ]
            return nn.Sequential(*layers)

        # انکودر
        self.enc1 = conv_block(in_channels, 128, kernel_size=3, stride=1, padding=1)
        self.enc2 = conv_block(128, 256, kernel_size=3, stride=2, padding=1)
        self.enc3 = conv_block(256, 512, kernel_size=3, stride=2, padding=1)
        self.enc4 = conv_block(512, 1024, kernel_size=3, stride=2, padding=1)

        # لایه میانی (Bottleneck)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(1024, 1024, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(p=0.3),  # Dropout2d در لایه میانی
        )

        # دکودر
        self.dec4 = upsample_block(1024, 512)
        self.dec3 = upsample_block(512 + 512, 256)
        self.dec2 = upsample_block(256 + 256, 128)
        self.dec1 = nn.Sequential(
            nn.Conv2d(128 + 128, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(p=0.3),
        )

        # لایه نهایی
        self.final = nn.Sequential(
            nn.Conv2d(64, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )

        # مقداردهی اولیه وزن‌ها
        def init_weights(m):
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu', a=0.2)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

        self.apply(init_weights)

    def forward(self, x):
        # انکودر
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        # لایه میانی
        b = self.bottleneck(e4)

        # دکودر با اتصالات میانبر
        d4 = self.dec4(b)
        if e3.shape[2:] != d4.shape[2:]:
            e3 = nn.functional.interpolate(e3, size=d4.shape[2:], mode='bilinear', align_corners=False)
        d4 = torch.cat([d4, e3], dim=1)
        d3 = self.dec3(d4)
        d3 = torch.cat([d3, e2], dim=1)
        d2 = self.dec2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        d1 = self.dec1(d2)

        # خروجی نهایی
        out = self.final(d1)
        return out

In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from PIL import Image
import os
import glob

class LicensePlateDataset(Dataset):
    def __init__(self, root_dir, labels_dict, transform=None, indices=None):
        self.root_dir = root_dir
        self.labels_dict = labels_dict
        self.transform = transform
        self.folders = sorted([f for f in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, f))])
        if indices is not None:
            self.folders = [self.folders[i] for i in indices]

    def __len__(self):
        return len(self.folders)

    def __getitem__(self, idx):
        try:
            folder = self.folders[idx]
            folder_path = os.path.join(self.root_dir, folder)
            img_files = sorted(glob.glob(os.path.join(folder_path, "*.png")))
            images = []
            for img_path in img_files[:5]:
                img = Image.open(img_path).convert('L')
                if self.transform:
                    img = self.transform(img)
                images.append(img.squeeze(0))
            images = torch.stack(images)
            label = self.labels_dict.get(f"{folder}.png", "")
            if not label:
                label = "UNKNOWN"
            return images, label
        except Exception as e:
            print(f"Error loading data for index {idx}: {e}")
            return torch.zeros((5, 64, 128)), "UNKNOWN"

In [ ]:
import random
import torch
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from PIL import Image
import numpy as np

class CustomBlackoutAndWhitePatch:
    def __init__(self, blackout_prob=0.2, patch_size_ratio=(0.2, 0.4)):
        """
        Custom transform to apply random blackout (replace with white) or set a random patch to white.
        Args:
            blackout_prob (float): Probability of blacking out the image and replacing with white.
            patch_size_ratio (tuple): Range of patch size ratio (relative to image size) for white patch.
        """
        self.blackout_prob = blackout_prob
        self.patch_size_ratio = patch_size_ratio

    def __call__(self, img):
        """
        Args:
            img (PIL.Image): Input image.
        Returns:
            PIL.Image: Image with either blackout (replaced with white) or a random patch set to white.
        """
        # Check if image should be blacked out
        if random.random() < self.blackout_prob:
            # Create a white image (same size as input)
            white_img = Image.fromarray(np.full((img.size[1], img.size[0]), 255, dtype=np.uint8), mode='L')
            return white_img

        # Otherwise, set a random patch to white
        width, height = img.size

        # Randomly select patch size (relative to image dimensions)
        patch_width = int(width * random.uniform(self.patch_size_ratio[0], self.patch_size_ratio[1]))
        patch_height = int(height * random.uniform(self.patch_size_ratio[0], self.patch_size_ratio[1]))

        # Ensure patch size is within bounds
        patch_width = min(patch_width, width)
        patch_height = min(patch_height, height)

        # Randomly select top-left corner of the patch
        x = random.randint(0, width - patch_width)
        y = random.randint(0, height - patch_height)

        # Create a white patch
        white_patch = Image.fromarray(np.full((patch_height, patch_width), 255, dtype=np.uint8), mode='L')
        # Create a copy of the image and paste the white patch
        img = img.copy()
        img.paste(white_patch, (x, y))
        return img

# Updated train_transform based on your original transform
train_transform = transforms.Compose([
    transforms.Resize((64, 128), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomAffine(
        degrees=(-5, 5),
        translate=(0.10, 0.10),
        scale=(0.95, 1.05),
        shear=(-5, 5),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.RandomRotation(degrees=(-5, 5), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.05),
    transforms.RandomAdjustSharpness(sharpness_factor=1.25, p=0.2),
    CustomBlackoutAndWhitePatch(blackout_prob=0.2, patch_size_ratio=(0.2, 0.4)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,)),
])

In [ ]:
import torch
torch.cuda.empty_cache()
print(torch.cuda.memory_reserved() / 1024**3)  # چک کن که کش خالی شده

0.1328125


In [ ]:
import torch
print(torch.cuda.is_available())  # چک کن که GPU در دسترسه
print(torch.cuda.memory_allocated() / 1024**3)  # حافظه تخصیص‌یافته (GB)
print(torch.cuda.memory_reserved() / 1024**3)  # حافظه رزروشده (کش) (GB)

True
0.07399463653564453
0.1328125


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torchmetrics.image import StructuralSimilarityIndexMeasure
import json
from itertools import cycle
import pandas as pd
import numpy as np
from PIL import Image
import os
import glob
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import matplotlib
import random

matplotlib.use('Agg')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 32
num_epochs = 2000
lr_g = 1e-4
n_critic = 2
lambda_gp = 10
lr_d = 2e-4
warmup_epochs = 0

class AddGaussianNoise(object):
    def __init__(self, mean=0.0, std_range=(0.02, 0.08)):
        self.mean = mean
        self.std_range = std_range

    def __call__(self, tensor):
        std = random.uniform(self.std_range[0], self.std_range[1])
        noise = torch.randn(tensor.size(), device=tensor.device) * std + self.mean
        return torch.clamp(tensor + noise, 0, 1)

class AddSaltAndPepperNoise(object):
    def __init__(self, prob=0.01):
        self.prob = prob

    def __call__(self, tensor):
        noise = torch.rand(tensor.size(), device=tensor.device)
        tensor = tensor.clone()
        tensor[noise < self.prob / 2] = 0
        tensor[noise > 1 - self.prob / 2] = 1
        return tensor


class HRDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = sorted(glob.glob(os.path.join(root_dir, "*.jpg")))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        try:
            img_path = self.images[idx]
            img = Image.open(img_path).convert('L')
            if self.transform:
                img = self.transform(img)
            return img
        except Exception as e:
            return torch.zeros((1, 64, 128))

hr_transform = transforms.Compose([
    transforms.Resize((64, 128), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.RandomAffine(
        degrees=(-4, 4),
        translate=(0.04, 0.04),
        scale=(0.96, 1.04),
        shear=(-3, 3),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.RandomPerspective(distortion_scale=0.05, p=0.1, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,)),
])

val_test_transform = transforms.Compose([
    transforms.Resize((64, 128), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5), std=(0.5)),
])

dataset_hr_path = "/content/dataset_hr"
dataset_lr_path = "/content/dataset_phase1"
labels_path = "/content/labels.csv"
vocab_path = "/content/vocab.json"
ocr_model_path = "/content/ocr_finetuned.pth"
output_dir = "/content/output"
os.makedirs(output_dir, exist_ok=True)

with open(vocab_path, 'r', encoding='utf-8') as f:
    vocab = json.load(f)
idx_to_char = {v: k for k, v in vocab.items()}

labels_df = pd.read_csv(labels_path)
labels_dict = dict(zip(labels_df['image'], labels_df['plate']))

total_size = len(os.listdir(dataset_lr_path))
indices = list(range(total_size))
random.shuffle(indices)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]
test_indices = indices[train_size + val_size:]

train_dataset = LicensePlateDataset(dataset_lr_path, labels_dict, transform=train_transform, indices=train_indices)
val_dataset = LicensePlateDataset(dataset_lr_path, labels_dict, transform=val_test_transform, indices=val_indices)
test_dataset = LicensePlateDataset(dataset_lr_path, labels_dict, transform=val_test_transform, indices=test_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    pin_memory=True,
    num_workers=2,
    prefetch_factor=4
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=True,
    pin_memory=True,
    num_workers=2,
    prefetch_factor=4
)
hr_loader = DataLoader(
    HRDataset(dataset_hr_path, transform=hr_transform),
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    pin_memory=True,
    num_workers=2,
    prefetch_factor=4
)

def test_rotation_transform(train_loader):
    data_iterator = iter(train_loader)
    try:
        images, _ = next(data_iterator)
    except ValueError:
        images = next(data_iterator)

In [ ]:
  def ctc_decode(output, idx_to_char):
      _, max_indices = torch.max(output, 2)
      decoded = []
      for indices in max_indices:
          seq = []
          prev = None
          for idx in indices:
              idx = idx.item()
              if idx != 0 and idx != prev:
                  seq.append(idx_to_char.get(idx, ""))
              prev = idx
          decoded.append("".join(seq))
      return decoded

  def compute_gradient_penalty(discriminator, real_samples, fake_samples, device):
      batch_size = real_samples.size(0)
      alpha = torch.rand(batch_size, 1, 1, 1, device=device, requires_grad=True)
      interpolates = alpha * real_samples + (1 - alpha) * fake_samples
      interpolates = interpolates.requires_grad_(True)
      d_interpolates = discriminator(interpolates)
      grad_outputs = torch.ones_like(d_interpolates, device=device)
      gradients = torch.autograd.grad(
          outputs=d_interpolates,
          inputs=interpolates,
          grad_outputs=grad_outputs,
          create_graph=True,
          retain_graph=True,
          only_inputs=True
      )[0]
      gradients = gradients.view(batch_size, -1)
      gradient_norm = gradients.norm(2, dim=1)
      gradient_penalty = ((gradient_norm - 1) ** 2).mean()
      return gradient_penalty

  def text_to_tensor(text, vocab, max_length=15):
      tensor = [vocab.get(char, 0) for char in text]
      tensor = tensor[:max_length] + [0] * (max_length - len(tensor))
      return torch.tensor(tensor, dtype=torch.long)

  def compute_char_accuracy(fake_images, labels, ocr_model, idx_to_char):
      char_correct = 0
      char_total = 0
      with torch.no_grad():
          ocr_output = ocr_model(fake_images)
          for i in range(ocr_output.size(0)):
              pred = ctc_decode(ocr_output[i:i+1], idx_to_char)[0]
              true_label = labels[i]
              min_len = min(len(pred), len(true_label))
              for j in range(min_len):
                  if pred[j] == true_label[j]:
                      char_correct += 1
              char_total += max(len(true_label), 1)
      return char_correct / char_total if char_total > 0 else 0

  def compute_gradient_norm(model, loss):
      model.zero_grad()
      loss.backward(retain_graph=True)
      grad_norm = 0
      for p in model.parameters():
          if p.grad is not None:
              grad_norm += p.grad.data.norm(2).item() ** 2
      return grad_norm ** 0.5

  def save_images_with_ocr(lr_images, fake_images, ocr_model, idx_to_char, epoch, batch_idx, output_dir):
      with torch.no_grad():
          ocr_model.eval()
          ocr_output = ocr_model(fake_images)
          ocr_preds = ctc_decode(ocr_output, idx_to_char)
          lr_ocr_outputs = [ocr_model(lr_images[:, i, :, :].unsqueeze(1)) for i in range(lr_images.size(1))]
          lr_ocr_preds = [ctc_decode(lr_ocr_output, idx_to_char) for lr_ocr_output in lr_ocr_outputs]

      for i in range(min(5, fake_images.size(0))):
          images = [lr_images[i, j, :, :].unsqueeze(0) for j in range(5)] + [fake_images[i]]
          grid = vutils.make_grid(images, normalize=True, nrow=6, padding=10, pad_value=1)
          grid_np = grid.permute(1, 2, 0).cpu().numpy()
          plt.figure(figsize=(20, 4))
          plt.imshow(grid_np, cmap='gray')
          plt.axis('off')
          for j in range(6):
              pred_text = lr_ocr_preds[j][i] if j < 5 else ocr_preds[i]
              plt.text(10 + j * (128 + 10), -5, f"OCR: {pred_text}", fontsize=10, color='red', ha='left', va='bottom')
          plt.savefig(os.path.join(output_dir, f'samples_epoch_{epoch+1}_batch_{i+1}.png'), bbox_inches='tight')
          plt.close()

  generator = Generator().to(device)
  discriminator = Discriminator().to(device)
  ocr_model = CRNN(num_classes=35, hidden_size=256).to(device)
  ocr_model.load_state_dict(torch.load(ocr_model_path, map_location=device))
  for param in ocr_model.parameters():
      param.requires_grad = False

  optimizer_g = optim.Adam(generator.parameters(), lr=lr_g, betas=(0.0, 0.9))
  optimizer_d = optim.Adam(discriminator.parameters(), lr=lr_d, betas=(0.0, 0.9))

  ctc_loss = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)
  mse_loss = nn.MSELoss()
  ssim = StructuralSimilarityIndexMeasure().to(device)

  d_real_history, d_fake_history, g_loss_history, d_loss_history = [], [], [], []
  val_char_acc_history, val_ocr_loss_history, val_mse_history, val_ssim_history = [], [], [], []
  adv_loss_history, ocr_loss_history, train_char_acc_history, train_ocr_loss_history = [], [], [], []
  lambda_adv_history, lambda_ocr_history = [], []
  adv_grad_norm_history, ocr_grad_norm_history = [], []

Total number of trainable parameters: 3,886,145


In [ ]:
from torchvision import transforms
import torch
from cleanfid import fid
from PIL import Image
import numpy as np
import os
import tempfile

def compute_fid(real_images, fake_images, device, batch_size=32):
    with tempfile.TemporaryDirectory() as real_dir, tempfile.TemporaryDirectory() as fake_dir:
        for i, img in enumerate(real_images):
            img = (img + 1.0) / 2.0 * 255
            if img.size(0) == 1:
                img = img.repeat(3, 1, 1)
            img = img.permute(1, 2, 0).cpu().numpy().astype(np.uint8)
            Image.fromarray(img).save(os.path.join(real_dir, f"real_{i}.png"))

        for i, img in enumerate(fake_images):
            img = (img + 1.0) / 2.0 * 255
            if img.size(0) == 1:
                img = img.repeat(3, 1, 1)
            img = img.permute(1, 2, 0).cpu().numpy().astype(np.uint8)
            Image.fromarray(img).save(os.path.join(fake_dir, f"fake_{i}.png"))

        fid_score = fid.compute_fid(real_dir, fake_dir, device=device, batch_size=batch_size)
    return fid_score

In [ ]:
import torch
import torch.optim as optim
import os
import json
import logging


def compute_gradient_norm(model, loss):
    model.zero_grad()
    loss.backward(retain_graph=True)
    grad_norm = 0
    for p in model.parameters():
        if p.grad is not None:
            grad_norm += p.grad.data.norm(2).item() ** 2
    return grad_norm ** 0.5

def compute_discriminator_grad_metrics(discriminator, real_images, fake_images, device):
    discriminator.zero_grad()
    real_validity = discriminator(real_images)
    fake_validity = discriminator(fake_images)
    gradient_penalty = compute_gradient_penalty(discriminator, real_images, fake_images, device)
    d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + lambda_gp * gradient_penalty
    d_loss.backward()

    grad_norms = []
    for p in discriminator.parameters():
        if p.grad is not None:
            grad_norms.append(torch.norm(p.grad, p=2).item())

    grad_norms = grad_norms if grad_norms else [0.0]
    return min(grad_norms), max(grad_norms), sum(grad_norms) / len(grad_norms)

def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if torch.cuda.is_available():
        torch.cuda.init()
        dummy = torch.tensor(0.0).to(device)
    generator.to(device)
    discriminator.to(device)
    ocr_model.to(device)

    print(f"Generator device: {next(generator.parameters()).device}")
    print(f"Discriminator device: {next(discriminator.parameters()).device}")
    print(f"OCR Model device: {next(ocr_model.parameters()).device}")
    print(f"GPU available: {torch.cuda.is_available()}, Device count: {torch.cuda.device_count()}, Current device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
    print(f"GPU memory allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB, Free memory: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0))/1e9:.2f} GB")
    print(f"Batch size: {train_loader.batch_size}")
    print(f"Generator parameters: {sum(p.numel() for p in generator.parameters()):,}")
    print(f"Discriminator parameters: {sum(p.numel() for p in discriminator.parameters()):,}")
    print(f"OCR Model parameters: {sum(p.numel() for p in ocr_model.parameters()):,}")
    lr_images, _ = next(iter(train_loader))
    lr_images = lr_images.to(device)
    real_images = next(iter(hr_loader)).to(device)
    print(f"lr_images device (first batch): {lr_images.device}")
    print(f"real_images device (first batch): {real_images.device}")
    print(f"lr_images shape (first batch): {lr_images.shape}")
    print(f"real_images shape (first batch): {real_images.shape}")
    if lr_images.device != device or real_images.device != device:
        print("Warning: Data is not on the same device as models! Check DataLoader or transforms.")
    print(f"n_critic: {n_critic}")
    print(f"HR transform: {hr_transform}")

    best_combined_metric = float('inf')
    fid_history = []
    d_output_max_history = []
    d_output_min_history = []
    d_grad_min_history = []
    d_grad_max_history = []
    d_grad_mean_history = []

    for epoch in range(num_epochs):
        generator.train()
        discriminator.train()
        ocr_model.train()
        total_d_loss, total_g_loss = 0, 0
        total_d_real_acc, total_d_fake_acc = 0, 0
        total_adv_loss, total_ocr_loss = 0, 0
        total_train_char_acc, total_train_ocr_loss = 0, 0
        total_lambda_adv, total_lambda_ocr = 0, 0
        total_adv_grad_norm, total_ocr_grad_norm = 0, 0
        total_fid = 0
        total_d_output_max, total_d_output_min = 0, 0
        total_d_grad_min, total_d_grad_max, total_d_grad_mean = 0, 0, 0
        total_batches = 0

        for i, (lr_images, labels) in enumerate(train_loader):
            lr_images = lr_images.to(device)  # [batch_size, 5, 64, 128]
            if i == 0 and epoch == 0:
                print(f"lr_images device: {lr_images.device}")
                print(f"lr_images shape: {lr_images.shape}")
            real_images = next(iter(hr_loader)).to(device)  # [batch_size, 1, 64, 128]
            batch_size = lr_images.size(0)

            # Warm-up phase: Train generator with only OCR loss for first 15 epochs
            if epoch < warmup_epochs:
                optimizer_g.zero_grad()
                fake_images = generator(lr_images)  # [batch_size, 1, 64, 128]
                ocr_output = ocr_model(fake_images)
                log_probs = ocr_output.permute(1, 0, 2).contiguous().log_softmax(2)
                input_lengths = torch.full((batch_size,), ocr_output.size(1), dtype=torch.long, device=device)
                target_lengths = torch.tensor([len(label) for label in labels], dtype=torch.long, device=device)
                targets = torch.cat([text_to_tensor(label, vocab, 15).unsqueeze(0) for label in labels]).to(device)
                ocr_loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)
                ocr_grad_norm = compute_gradient_norm(generator, ocr_loss)
                g_loss = ocr_loss
                g_loss.backward()
                torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
                optimizer_g.step()
                total_g_loss += g_loss.item()
                total_ocr_loss += ocr_loss.item()
                total_ocr_grad_norm += ocr_grad_norm
                total_lambda_ocr += 1.0  # lambda_ocr = 1.0 during warmup
            else:
                # Train discriminator
                for _ in range(n_critic):
                    optimizer_d.zero_grad()
                    fake_images = generator(lr_images).detach()  # [batch_size, 1, 64, 128]
                    real_validity = discriminator(real_images)
                    fake_validity = discriminator(fake_images)
                    gradient_penalty = compute_gradient_penalty(discriminator, real_images, fake_images, device)
                    d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + lambda_gp * gradient_penalty
                    d_loss.backward()
                    torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
                    optimizer_d.step()

                    d_grad_min, d_grad_max, d_grad_mean = compute_discriminator_grad_metrics(discriminator, real_images, fake_images, device)
                    total_d_grad_min += d_grad_min
                    total_d_grad_max += d_grad_max
                    total_d_grad_mean += d_grad_mean

                    total_d_loss += d_loss.item()
                    total_d_real_acc += real_validity.mean().item()
                    total_d_fake_acc += fake_validity.mean().item()
                    total_d_output_max += real_validity.max().item() + fake_validity.max().item()
                    total_d_output_min += real_validity.min().item() + fake_validity.min().item()

                # Train generator with combined loss
                optimizer_g.zero_grad()
                fake_images = generator(lr_images)  # [batch_size, 1, 64, 128]
                adv_loss = -torch.mean(discriminator(fake_images))
                ocr_output = ocr_model(fake_images)
                log_probs = ocr_output.permute(1, 0, 2).contiguous().log_softmax(2)
                input_lengths = torch.full((batch_size,), ocr_output.size(1), dtype=torch.long, device=device)
                target_lengths = torch.tensor([len(label) for label in labels], dtype=torch.long, device=device)
                targets = torch.cat([text_to_tensor(label, vocab, 15).unsqueeze(0) for label in labels]).to(device)
                ocr_loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)
                adv_grad_norm = compute_gradient_norm(generator, adv_loss)
                ocr_grad_norm = compute_gradient_norm(generator, ocr_loss)
                lambda_adv = 0.5
                lambda_ocr = 0.5
                g_loss = lambda_adv * adv_loss + lambda_ocr * ocr_loss
                g_loss.backward()
                torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
                optimizer_g.step()
                total_g_loss += g_loss.item()
                total_adv_loss += adv_loss.item()
                total_ocr_loss += ocr_loss.item()
                total_lambda_adv += lambda_adv
                total_lambda_ocr += lambda_ocr
                total_adv_grad_norm += adv_grad_norm
                total_ocr_grad_norm += ocr_grad_norm

            total_batches += 1

            with torch.no_grad():
                ocr_model.eval()
                train_char_acc = compute_char_accuracy(fake_images, labels, ocr_model, idx_to_char)
                ocr_model.train()
                total_train_char_acc += train_char_acc
                total_train_ocr_loss += ocr_loss.item()

            if (epoch % 50) == 0:
                with torch.no_grad():
                    ocr_model.eval()
                    fake_images = generator(lr_images)  # [batch_size, 1, 64, 128]
                    save_images_with_ocr(lr_images, fake_images, ocr_model, idx_to_char, epoch, i, output_dir)
                    ocr_model.train()

        generator.eval()
        total_val_ocr_loss, total_val_char_acc, total_val_mse, total_val_ssim, total_val_fid = 0, 0, 0, 0, 0
        val_batches = 0
        all_fake_images = []
        all_real_images = []
        with torch.no_grad():
            for lr_images, labels in val_loader:
                lr_images = lr_images.to(device)  # [batch_size, 5, 64, 128]
                fake_images = generator(lr_images)  # [batch_size, 1, 64, 128]
                real_images = next(iter(hr_loader)).to(device)  # [batch_size, 1, 64, 128]
                ocr_model.eval()
                ocr_output = ocr_model(fake_images)
                log_probs = ocr_output.permute(1, 0, 2).contiguous().log_softmax(2)
                input_lengths = torch.full((batch_size,), ocr_output.size(1), dtype=torch.long, device=device)
                target_lengths = torch.tensor([len(label) for label in labels], dtype=torch.long, device=device)
                targets = torch.cat([text_to_tensor(label, vocab, 15).unsqueeze(0) for label in labels]).to(device)
                val_ocr_loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)
                val_char_acc = compute_char_accuracy(fake_images, labels, ocr_model, idx_to_char)
                val_mse = mse_loss(fake_images, real_images)
                val_ssim = ssim(fake_images, real_images)

                # Collect images for FID calculation
                all_fake_images.append(fake_images)
                all_real_images.append(real_images)

                total_val_ocr_loss += val_ocr_loss.item()
                total_val_char_acc += val_char_acc
                total_val_mse += val_mse.item()
                total_val_ssim += val_ssim.item()
                val_batches += 1

            if val_batches > 0:
                # Compute FID every 50 epochs using all collected images
                if epoch % 50 == 0:
                    all_fake_images = torch.cat(all_fake_images, dim=0)
                    all_real_images = torch.cat(all_real_images, dim=0)
                    val_fid = compute_fid(all_real_images, all_fake_images, device)
                    total_val_fid = val_fid * val_batches  # Scale to maintain averaging logic
                else:
                    val_fid = 0.0
                    total_val_fid = 0.0

                avg_val_ocr_loss = total_val_ocr_loss / val_batches
                avg_val_char_acc = total_val_char_acc / val_batches
                avg_val_mse = total_val_mse / val_batches
                avg_val_ssim = total_val_ssim / val_batches
                avg_val_fid = total_val_fid / val_batches if epoch % 10 == 0 else 0.0

        avg_d_loss = total_d_loss / (total_batches * n_critic) if total_batches > 0 and epoch >= warmup_epochs else 0
        avg_g_loss = total_g_loss / total_batches if total_batches > 0 else 0
        avg_d_real_acc = total_d_real_acc / (total_batches * n_critic) if total_batches > 0 and epoch >= warmup_epochs else 0
        avg_d_fake_acc = total_d_fake_acc / (total_batches * n_critic) if total_batches > 0 and epoch >= warmup_epochs else 0
        avg_adv_loss = total_adv_loss / total_batches if total_batches > 0 and epoch >= warmup_epochs else 0
        avg_ocr_loss = total_ocr_loss / total_batches if total_batches > 0 else 0
        avg_lambda_adv = total_lambda_adv / total_batches if total_batches > 0 and epoch >= warmup_epochs else 0
        avg_lambda_ocr = total_lambda_ocr / total_batches if total_batches > 0 else 1.0 if epoch < warmup_epochs else 0.3
        avg_adv_grad_norm = total_adv_grad_norm / total_batches if total_batches > 0 and epoch >= warmup_epochs else 0
        avg_ocr_grad_norm = total_ocr_grad_norm / total_batches if total_batches > 0 else 0
        avg_train_char_acc = total_train_char_acc / total_batches if total_batches > 0 else 0
        avg_train_ocr_loss = total_train_ocr_loss / total_batches if total_batches > 0 else 0
        avg_val_ocr_loss = total_val_ocr_loss / val_batches if val_batches > 0 else float('inf')
        avg_val_char_acc = total_val_char_acc / val_batches if val_batches > 0 else 0
        avg_val_mse = total_val_mse / val_batches if val_batches > 0 else 0
        avg_val_ssim = total_val_ssim / val_batches if val_batches > 0 else 0
        avg_val_fid = total_val_fid / val_batches if val_batches > 0 and epoch % 10 == 0 else 0
        avg_d_output_max = total_d_output_max / (total_batches * 2) if total_batches > 0 and epoch >= warmup_epochs else 0
        avg_d_output_min = total_d_output_min / (total_batches * 2) if total_batches > 0 and epoch >= warmup_epochs else 0
        avg_d_grad_min = total_d_grad_min / (total_batches * n_critic) if total_batches > 0 and epoch >= warmup_epochs else 0
        avg_d_grad_max = total_d_grad_max / (total_batches * n_critic) if total_batches > 0 and epoch >= warmup_epochs else 0
        avg_d_grad_mean = total_d_grad_mean / (total_batches * n_critic) if total_batches > 0 and epoch >= warmup_epochs else 0

        d_real_history.append(avg_d_real_acc)
        d_fake_history.append(avg_d_fake_acc)
        g_loss_history.append(avg_g_loss)
        d_loss_history.append(avg_d_loss)
        val_char_acc_history.append(avg_val_char_acc)
        val_ocr_loss_history.append(avg_val_ocr_loss)
        val_mse_history.append(avg_val_mse)
        val_ssim_history.append(avg_val_ssim)
        adv_loss_history.append(avg_adv_loss)
        ocr_loss_history.append(avg_ocr_loss)
        train_char_acc_history.append(avg_train_char_acc)
        train_ocr_loss_history.append(avg_train_ocr_loss)
        lambda_adv_history.append(avg_lambda_adv)
        lambda_ocr_history.append(avg_lambda_ocr)
        adv_grad_norm_history.append(avg_adv_grad_norm)
        ocr_grad_norm_history.append(avg_ocr_grad_norm)
        fid_history.append(avg_val_fid)
        d_output_max_history.append(avg_d_output_max)
        d_output_min_history.append(avg_d_output_min)
        d_grad_min_history.append(avg_d_grad_min)
        d_grad_max_history.append(avg_d_grad_max)
        d_grad_mean_history.append(avg_d_grad_mean)

        metrics = {
            'epoch': epoch + 1,
            'd_real_acc': avg_d_real_acc,
            'd_fake_acc': avg_d_fake_acc,
            'g_loss': avg_g_loss,
            'd_loss': avg_d_loss,
            'adv_loss': avg_adv_loss,
            'ocr_loss': avg_ocr_loss,
            'lambda_adv': avg_lambda_adv,
            'lambda_ocr': avg_lambda_ocr,
            'adv_grad_norm': avg_adv_grad_norm,
            'ocr_grad_norm': avg_ocr_grad_norm,
            'train_char_acc': avg_train_char_acc,
            'train_ocr_loss': avg_train_ocr_loss,
            'val_char_acc': avg_val_char_acc,
            'val_ocr_loss': avg_val_ocr_loss,
            'val_mse': avg_val_mse,
            'val_ssim': avg_val_ssim,
            'val_fid': avg_val_fid,
            'd_output_max': avg_d_output_max,
            'd_output_min': avg_d_output_min,
            'd_grad_min': avg_d_grad_min,
            'd_grad_max': avg_d_grad_max,
            'd_grad_mean': avg_d_grad_mean
        }
        with open(os.path.join(output_dir, 'metrics.json'), 'a') as f:
            json.dump(metrics, f)
            f.write('\n')

        history = {
            'd_real_history': d_real_history,
            'd_fake_history': d_fake_history,
            'g_loss_history': g_loss_history,
            'd_loss_history': d_loss_history,
            'adv_loss_history': adv_loss_history,
            'ocr_loss_history': ocr_loss_history,
            'train_char_acc_history': train_char_acc_history,
            'train_ocr_loss_history': train_ocr_loss_history,
            'val_char_acc_history': val_char_acc_history,
            'val_ocr_loss_history': val_ocr_loss_history,
            'val_mse_history': val_mse_history,
            'val_ssim_history': val_ssim_history,
            'lambda_adv_history': lambda_adv_history,
            'lambda_ocr_history': lambda_ocr_history,
            'adv_grad_norm_history': adv_grad_norm_history,
            'ocr_grad_norm_history': ocr_grad_norm_history,
            'fid_history': fid_history,
            'd_output_max_history': d_output_max_history,
            'd_output_min_history': d_output_min_history,
            'd_grad_min_history': d_grad_min_history,
            'd_grad_max_history': d_grad_max_history,
            'd_grad_mean_history': d_grad_mean_history
        }
        with open(os.path.join(output_dir, 'history.json'), 'w') as f:
            json.dump(history, f)

        combined_metric = avg_val_ocr_loss + 10 * (1 - avg_val_char_acc)

        if combined_metric < best_combined_metric:
            best_combined_metric = combined_metric
            torch.save({
                'generator_state_dict': generator.state_dict(),
                'discriminator_state_dict': discriminator.state_dict(),
                'optimizer_g_state_dict': optimizer_g.state_dict(),
                'optimizer_d_state_dict': optimizer_d.state_dict(),
                'epoch': epoch + 1,
                'val_ocr_loss': avg_val_ocr_loss,
                'val_char_acc': avg_val_char_acc,
                'combined_metric': combined_metric,
                'val_fid': avg_val_fid
            }, os.path.join(output_dir, 'best_phase1m1.pth'))
            print(f"Model saved at epoch {epoch + 1} with combined metric {combined_metric:.4f}, FID: {avg_val_fid:.4f}")

        if len(d_real_history) >= 20 and epoch >= warmup_epochs:
            recent_d_real = d_real_history[-20:]
            recent_d_fake = d_fake_history[-20:]
            if all(abs(r - f) < 0.01 for r, f in zip(recent_d_real, recent_d_fake)):
                print("Discriminator outputs converged! Stopping training.")
                break

        print((
            f"Epoch: {epoch + 1}\n"
            f"D Real Acc: {avg_d_real_acc:.4f}\n"
            f"D Fake Acc: {avg_d_fake_acc:.4f}\n"
            f"d: {(avg_d_fake_acc - avg_d_real_acc):.4f}\n"
            f"G Loss: {avg_g_loss:.4f}\n"
            f"D Loss: {avg_d_loss:.4f}\n"
            f"Adv Loss: {avg_adv_loss:.4f}\n"
            f"OCR Loss: {avg_ocr_loss:.4f}\n"
            f"Lambda Adv: {avg_lambda_adv:.4f}\n"
            f"Lambda OCR: {avg_lambda_ocr:.4f}\n"
            f"Adv Grad Norm: {avg_adv_grad_norm:.4f}\n"
            f"OCR Grad Norm: {avg_ocr_grad_norm:.4f}\n"
            f"Train Char Acc: {avg_train_char_acc:.4f}\n"
            f"Train OCR Loss: {avg_train_ocr_loss:.4f}\n"
            f"Val Char Acc: {avg_val_char_acc:.4f}\n"
            f"Val OCR Loss: {avg_val_ocr_loss:.4f}\n"
            f"Val MSE: {avg_val_mse:.4f}\n"
            f"Val SSIM: {avg_val_ssim:.4f}\n"
            f"Val FID: {avg_val_fid:.4f}\n"
            f"D Output Max: {avg_d_output_max:.4f}\n"
            f"D Output Min: {avg_d_output_min:.4f}\n"
            f"D Grad Min: {avg_d_grad_min:.4f}\n"
            f"D Grad Max: {avg_d_grad_max:.4f}\n"
            f"D Grad Mean: {avg_d_grad_mean:.4f}\n"
            "--------------------------------------------------"
        ), flush=True)

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    discriminator.to(device)
    generator.to(device)
    ocr_model = ocr_model.to(device)
    train()

Generator device: cuda:0
Discriminator device: cuda:0
OCR Model device: cuda:0
GPU available: True, Device count: 1, Current device: Tesla T4
GPU memory allocated: 0.55 GB, Free memory: 8.59 GB
Batch size: 32
Generator parameters: 23,460,737
Discriminator parameters: 3,886,145
OCR Model parameters: 8,721,699
lr_images device (first batch): cuda:0
real_images device (first batch): cuda:0
lr_images shape (first batch): torch.Size([32, 5, 64, 128])
real_images shape (first batch): torch.Size([32, 1, 64, 128])
n_critic: 2
HR transform: Compose(
    Resize(size=(64, 128), interpolation=bicubic, max_size=None, antialias=True)
    ColorJitter(brightness=(0.9, 1.1), contrast=(0.9, 1.1), saturation=None, hue=None)
    RandomAffine(degrees=[-4.0, 4.0], translate=(0.04, 0.04), scale=(0.96, 1.04), shear=[-3.0, 3.0], interpolation=bilinear)
    RandomPerspective(p=0.1)
    ToTensor()
    Normalize(mean=(0.5,), std=(0.5,))
)
lr_images device: cuda:0
lr_images shape: torch.Size([32, 5, 64, 128])


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


compute FID between two folders
Found 32 images in the folder /tmp/tmp5r34gimt


FID tmp5r34gimt : 100%|██████████| 1/1 [00:01<00:00,  1.48s/it]


Found 32 images in the folder /tmp/tmpm0mciifi


...

FID tmpgf20o9u1 : 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]


Found 32 images in the folder /tmp/tmpvs4_0peh


FID tmpvs4_0peh : 100%|██████████| 1/1 [00:04<00:00,  4.32s/it]


Epoch: 301
D Real Acc: -2.2259
D Fake Acc: -6.6281
d: -4.4022
G Loss: 4.2896
D Loss: -4.0149
Adv Loss: 7.1926
OCR Loss: 1.3866
Lambda Adv: 0.5000
Lambda OCR: 0.5000
Adv Grad Norm: 18.0639
OCR Grad Norm: 7.5409
Train Char Acc: 0.7331
Train OCR Loss: 1.3866
Val Char Acc: 0.9570
Val OCR Loss: 0.2421
Val MSE: 0.4541
Val SSIM: 0.0813
Val FID: 179.7852
D Output Max: 6.4898
D Output Min: -26.2272
D Grad Min: 0.0000
D Grad Max: 3.4335
D Grad Mean: 0.9849
--------------------------------------------------
Epoch: 302
D Real Acc: -2.1826
D Fake Acc: -6.7605
d: -4.5780
G Loss: 4.0183
D Loss: -4.1562
Adv Loss: 6.7493
OCR Loss: 1.2873
Lambda Adv: 0.5000
Lambda OCR: 0.5000
Adv Grad Norm: 12.6474
OCR Grad Norm: 6.4439
Train Char Acc: 0.7402
Train OCR Loss: 1.2873
Val Char Acc: 0.9570
Val OCR Loss: 0.2435
Val MSE: 0.4245
Val SSIM: 0.0864
Val FID: 0.0000
D Output Max: 8.3840
D Output Min: -25.6397
D Grad Min: 0.0000
D Grad Max: 3.2664
D Grad Mean: 0.9684
-------------------------------------------------

KeyboardInterrupt: 

90min

In [ ]:
import json
import matplotlib.pyplot as plt
import os
import numpy as np

output_dir = "/content/output"

with open(os.path.join(output_dir, 'history.json'), 'r') as f:
    history = json.load(f)

d_real_history = history['d_real_history']
d_fake_history = history['d_fake_history']
g_loss_history = history['g_loss_history']
d_loss_history = history['d_loss_history']
adv_loss_history = history['adv_loss_history']
ocr_loss_history = history['ocr_loss_history']
train_char_acc_history = history['train_char_acc_history']
train_ocr_loss_history = history['train_ocr_loss_history']
val_char_acc_history = history['val_char_acc_history']
val_ocr_loss_history = history['val_ocr_loss_history']
val_mse_history = history['val_mse_history']
val_ssim_history = history['val_ssim_history']
fid_history = history['fid_history']
lambda_adv_history = history['lambda_adv_history']
lambda_ocr_history = history['lambda_ocr_history']
adv_grad_norm_history = history['adv_grad_norm_history']
ocr_grad_norm_history = history['ocr_grad_norm_history']
d_output_max_history = history['d_output_max_history']
d_output_min_history = history['d_output_min_history']
d_grad_min_history = history['d_grad_min_history']
d_grad_max_history = history['d_grad_max_history']
d_grad_mean_history = history['d_grad_mean_history']

# Calculate ratio of D Fake Acc to D Real Acc
ratio_history = [d_fake - d_real for d_real, d_fake in zip(d_real_history, d_fake_history)]

plt.figure(figsize=(15, 20))

# 1. Discriminator Gradient Norms (High Priority)
plt.subplot(5, 4, 1)
plt.plot(d_grad_mean_history, label='D Grad Mean', color='green')
plt.xlabel('Epoch')
plt.ylabel('Discriminator Grad Mean')
plt.legend()

plt.subplot(5, 4, 2)
plt.plot(d_grad_min_history, label='D Grad Min', color='blue')
plt.xlabel('Epoch')
plt.ylabel('Discriminator Grad Min')
plt.legend()

plt.subplot(5, 4, 3)
plt.plot(d_grad_max_history, label='D Grad Max', color='red')
plt.xlabel('Epoch')
plt.ylabel('Discriminator Grad Max')
plt.legend()

# 2. Losses (High Priority)
plt.subplot(5, 4, 4)
plt.plot(g_loss_history, label='G Loss', color='green')
plt.plot(d_loss_history, label='D Loss', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(5, 4, 5)
plt.plot(adv_loss_history, label='Adv Loss', color='purple')
plt.xlabel('Epoch')
plt.ylabel('Adversarial Loss')
plt.legend()

plt.subplot(5, 4, 6)
plt.plot(ocr_loss_history, label='OCR Loss', color='brown')
plt.xlabel('Epoch')
plt.ylabel('OCR Loss')
plt.legend()

# 3. FID (High Priority)
plt.subplot(5, 4, 7)
plt.plot(fid_history, label='Val FID', color='navy')
plt.xlabel('Epoch')
plt.ylabel('FID')
plt.legend()

# 4. OCR Accuracy
plt.subplot(5, 4, 8)
plt.plot(train_char_acc_history, label='Train Char Acc', color='cyan')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(5, 4, 9)
plt.plot(val_char_acc_history, label='Val Char Acc', color='purple')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# 5. Validation Losses
plt.subplot(5, 4, 10)
plt.plot(train_ocr_loss_history, label='Train OCR Loss', color='brown')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(5, 4, 11)
plt.plot(val_ocr_loss_history, label='Val OCR Loss', color='magenta')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# 6. SSIM and MSE
plt.subplot(5, 4, 12)
plt.plot(val_ssim_history, label='Val SSIM', color='pink')
plt.xlabel('Epoch')
plt.ylabel('SSIM')
plt.legend()

plt.subplot(5, 4, 13)
plt.plot(val_mse_history, label='Val MSE', color='teal')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.plot()

# 7. Discriminator Outputs
plt.subplot(5, 4, 14)
plt.plot(d_real_history, label='D(E(real))', color='blue')
plt.plot(d_fake_history, label='D(E(fake))', color='red')
plt.xlabel('Epoch')
plt.ylabel('Discriminator Output')
plt.legend()

plt.subplot(5, 4, 15)
plt.plot(d_output_max_history, label='D Output Max', color='darkgreen')
plt.xlabel('Epoch')
plt.ylabel('Max Discriminator Output')
plt.legend()

plt.subplot(5, 4, 16)
plt.plot(d_output_min_history, label='D Output Min', color='darkred')
plt.xlabel('Epoch')
plt.ylabel('Min Discriminator Output')
plt.legend()

plt.subplot(5, 4, 17)
plt.plot(ratio_history, label='Wasserstein distance', color='cyan')
plt.xlabel('Epoch')
plt.ylabel('Wasserstein')
plt.legend()

# 8. Lambda Metrics (Low Priority)
plt.subplot(5, 4, 18)
plt.plot(lambda_adv_history, label='Lambda Adv', color='blue')
plt.xlabel('Epoch')
plt.ylabel('Lambda Adv')
plt.legend()

plt.subplot(5, 4, 19)
plt.plot(lambda_ocr_history, label='Lambda OCR', color='red')
plt.xlabel('Epoch')
plt.ylabel('Lambda OCR')
plt.legend()

# 9. Generator Gradient Norms
plt.subplot(5, 4, 20)
plt.plot(adv_grad_norm_history, label='Adv Grad Norm', color='green')
plt.plot(ocr_grad_norm_history, label='OCR Grad Norm', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Generator Grad Norm')
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'metrics_plot.png'))
plt.close()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import json
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator_model_path = "output/best_phase1m1.pth"
ocr_model_path = "ocr_finetuned.pth"
output_dir = "/output"
os.makedirs(output_dir, exist_ok=True)

with open("vocab.json", 'r', encoding='utf-8') as f:
    vocab = json.load(f)
idx_to_char = {v: k for k, v in vocab.items()}


import torch
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, in_channels=5):
        super(Generator, self).__init__()

        def conv_block(in_channels, out_channels, kernel_size=3, stride=1, padding=1, use_bn=True):
            layers = [
                nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=not use_bn),
                nn.BatchNorm2d(out_channels) if use_bn else nn.Identity(),
                nn.LeakyReLU(0.2, inplace=True),
                nn.Dropout2d(p=0.3),  # اضافه کردن Dropout2d برای فیوژن بهتر
            ]
            return nn.Sequential(*layers)

        def upsample_block(in_channels, out_channels, kernel_size=3, stride=1, padding=1, use_bn=True):
            layers = [
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
                nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=not use_bn),
                nn.BatchNorm2d(out_channels) if use_bn else nn.Identity(),
                nn.LeakyReLU(0.2, inplace=True),
                nn.Dropout2d(p=0.3),  # اضافه کردن Dropout2d
            ]
            return nn.Sequential(*layers)

        # انکودر
        self.enc1 = conv_block(in_channels, 128, kernel_size=3, stride=1, padding=1)
        self.enc2 = conv_block(128, 256, kernel_size=3, stride=2, padding=1)
        self.enc3 = conv_block(256, 512, kernel_size=3, stride=2, padding=1)
        self.enc4 = conv_block(512, 1024, kernel_size=3, stride=2, padding=1)

        # لایه میانی (Bottleneck)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(1024, 1024, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(p=0.3),  # Dropout2d در لایه میانی
        )

        # دکودر
        self.dec4 = upsample_block(1024, 512)
        self.dec3 = upsample_block(512 + 512, 256)
        self.dec2 = upsample_block(256 + 256, 128)
        self.dec1 = nn.Sequential(
            nn.Conv2d(128 + 128, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(p=0.3),
        )

        # لایه نهایی
        self.final = nn.Sequential(
            nn.Conv2d(64, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )

        # مقداردهی اولیه وزن‌ها
        def init_weights(m):
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu', a=0.2)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

        self.apply(init_weights)

    def forward(self, x):
        # انکودر
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        # لایه میانی
        b = self.bottleneck(e4)

        # دکودر با اتصالات میانبر
        d4 = self.dec4(b)
        if e3.shape[2:] != d4.shape[2:]:
            e3 = nn.functional.interpolate(e3, size=d4.shape[2:], mode='bilinear', align_corners=False)
        d4 = torch.cat([d4, e3], dim=1)
        d3 = self.dec3(d4)
        d3 = torch.cat([d3, e2], dim=1)
        d2 = self.dec2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        d1 = self.dec1(d2)

        # خروجی نهایی
        out = self.final(d1)
        return out


def ctc_decode(output, idx_to_char):
    _, max_indices = torch.max(output, 2)
    decoded = []
    for indices in max_indices:
        seq = []
        prev = None
        for idx in indices:
            idx = idx.item()
            if idx != 0 and idx != prev:
                seq.append(idx_to_char.get(idx, ""))
            prev = idx
        decoded.append("".join(seq))
    return decoded

def compute_char_accuracy(fake_images, labels, ocr_model, idx_to_char):
    char_correct = 0
    char_total = 0
    with torch.no_grad():
        ocr_output = ocr_model(fake_images)
        for i in range(ocr_output.size(0)):
            pred = ctc_decode(ocr_output[i:i+1], idx_to_char)[0]
            true_label = labels[i]
            min_len = min(len(pred), len(true_label))
            for j in range(min_len):
                if pred[j] == true_label[j]:
                    char_correct += 1
            char_total += max(len(true_label), 1)
    return char_correct / char_total if char_total > 0 else 0

def compute_plate_accuracy(fake_images, labels, ocr_model, idx_to_char):
    plate_correct = 0
    plate_total = len(labels)
    with torch.no_grad():
        ocr_output = ocr_model(fake_images)
        for i in range(ocr_output.size(0)):
            pred = ctc_decode(ocr_output[i:i+1], idx_to_char)[0]
            true_label = labels[i]
            if pred == true_label:
                plate_correct += 1
    return plate_correct / plate_total if plate_total > 0 else 0

test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, drop_last=False)

generator = Generator().to(device)
checkpoint = torch.load(generator_model_path, map_location=device, weights_only=False)
generator.load_state_dict(checkpoint['generator_state_dict'])
generator.eval()

ocr_model = CRNN(num_classes=35, hidden_size=256).to(device)
ocr_model.load_state_dict(torch.load(ocr_model_path, map_location=device, weights_only=False))
ocr_model.eval()

def evaluate():
    total_char_acc = 0
    total_plate_acc = 0
    total_batches = 0
    with torch.no_grad():
        for lr_images, labels in test_loader:
            lr_images = lr_images.to(device)  # [batch_size, 5, 64, 128]
            print(f"lr_images shape: {lr_images.shape}")
            fake_images = generator(lr_images)  # [batch_size, 1, 64, 128]
            print(f"fake_images shape: {fake_images.shape}")
            char_acc = compute_char_accuracy(fake_images, labels, ocr_model, idx_to_char)
            plate_acc = compute_plate_accuracy(fake_images, labels, ocr_model, idx_to_char)
            total_char_acc += char_acc
            total_plate_acc += plate_acc
            total_batches += 1
    avg_char_acc = total_char_acc / total_batches if total_batches > 0 else 0
    avg_plate_acc = total_plate_acc / total_batches if total_batches > 0 else 0
    print(f"Test Character Accuracy: {avg_char_acc:.4f}")
    print(f"Test Plate Accuracy: {avg_plate_acc:.4f}")
    eval_results = {
        'test_char_accuracy': avg_char_acc,
        'test_plate_accuracy': avg_plate_acc
    }
    with open(os.path.join(output_dir, 'evaluation_results.json'), 'w') as f:
        json.dump(eval_results, f, indent=4)

if __name__ == "__main__":
    evaluate()

lr_images shape: torch.Size([8, 5, 64, 128])
fake_images shape: torch.Size([8, 1, 64, 128])
lr_images shape: torch.Size([8, 5, 64, 128])
fake_images shape: torch.Size([8, 1, 64, 128])
lr_images shape: torch.Size([8, 5, 64, 128])
fake_images shape: torch.Size([8, 1, 64, 128])
lr_images shape: torch.Size([8, 5, 64, 128])
fake_images shape: torch.Size([8, 1, 64, 128])
lr_images shape: torch.Size([8, 5, 64, 128])
fake_images shape: torch.Size([8, 1, 64, 128])
lr_images shape: torch.Size([5, 5, 64, 128])
fake_images shape: torch.Size([5, 1, 64, 128])
Test Character Accuracy: 0.8974
Test Plate Accuracy: 0.6500


Test Character Accuracy: 0.9865
Test Plate Accuracy: 0.9250

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 8
num_epochs = 400
lr_g = 1e-4
lr_d = 8e-5
lambda_gp = 10
n_critic = 4


تازه هنوز به تعادل هم نرسیده بود


In [ ]:
1_1
2_1